<a href="https://colab.research.google.com/github/roopaam/MB-Proxy/blob/main/ACSIncome_Tradeoff_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from folktables import ACSDataSource, ACSIncome

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import KBinsDiscretizer, OneHotEncoder
from sklearn.impute import SimpleImputer
from scipy.stats import ttest_rel

from fairlearn.metrics import demographic_parity_difference


def load_acs_income_df(survey_year="2018", horizon="1-Year", state='CA'):
    data_source = ACSDataSource(survey_year=survey_year, horizon=horizon, survey="person")
    acs_data = data_source.get_data(states=[state] if state else None, download=True)

    X, y, _ = ACSIncome.df_to_numpy(acs_data)
    df = pd.DataFrame(X, columns=ACSIncome.features)
    df["income"] = y.astype(int)
    return df


def build_pipeline(feature_cols, continuous_cols):
    cont_cols = [c for c in feature_cols if c in continuous_cols]
    cat_cols = [c for c in feature_cols if c not in continuous_cols]

    preprocess = ColumnTransformer(
        transformers=[
            ("cont", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("binner", KBinsDiscretizer(n_bins=5, encode="onehot-dense", strategy="quantile")),
            ]), cont_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), cat_cols),
        ],
        remainder="drop"
    )

    model = LogisticRegression(max_iter=2000, solver="lbfgs")

    return Pipeline([("preprocess", preprocess), ("model", model)])


def evaluate_tradeoff_repeated_splits(
    df,
    target_col,
    sensitive_col,
    mb_features,
    proxy_features,
    n_splits=30,
    test_size=0.30,
    random_state=42,
):
    # Define feature sets
    fs_mb = sorted(set(mb_features + [sensitive_col]))
    fs_fair = [f for f in mb_features if f not in set(proxy_features) and f != sensitive_col]

    feature_sets = {"MB_Shield": fs_mb, "Fair_Filter": fs_fair}

    # Identify continuous columns
    continuous_cols = [c for c in df.columns if c not in [target_col] and pd.api.types.is_numeric_dtype(df[c])]

    X = df.drop(columns=[target_col])
    y = df[target_col].to_numpy()
    s = df[sensitive_col].to_numpy()

    splitter = StratifiedShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=random_state)

    rows = []
    for split_id, (tr, te) in enumerate(splitter.split(X, y), start=1):
        X_train_full = X.iloc[tr].copy()
        X_test_full = X.iloc[te].copy()
        y_train = y[tr]
        y_test = y[te]
        s_test = s[te]

        for cfg_name, feat_cols in feature_sets.items():
            pipe = build_pipeline(feat_cols, continuous_cols)

            pipe.fit(X_train_full[feat_cols], y_train)
            y_pred = pipe.predict(X_test_full[feat_cols])

            acc = accuracy_score(y_test, y_pred)
            dpd = demographic_parity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=s_test)

            rows.append({
                "split": split_id,
                "config": cfg_name,
                "accuracy": acc,
                "dpd": dpd,
                "n_features": len(feat_cols),
            })

    results = pd.DataFrame(rows)

    # Paired differences
    acc_p = results.pivot(index="split", columns="config", values="accuracy")
    dpd_p = results.pivot(index="split", columns="config", values="dpd")

    acc_diff = acc_p["Fair_Filter"] - acc_p["MB_Shield"]
    dpd_diff = dpd_p["Fair_Filter"] - dpd_p["MB_Shield"]

    # Paired t-tests
    acc_t = ttest_rel(acc_p["Fair_Filter"], acc_p["MB_Shield"])
    dpd_t = ttest_rel(dpd_p["Fair_Filter"], dpd_p["MB_Shield"])

    summary = pd.DataFrame([
        {
            "config": "MB_Shield",
            "accuracy_mean": results.loc[results["config"] == "MB_Shield", "accuracy"].mean(),
            "accuracy_std": results.loc[results["config"] == "MB_Shield", "accuracy"].std(ddof=1),
            "dpd_mean": results.loc[results["config"] == "MB_Shield", "dpd"].mean(),
            "dpd_std": results.loc[results["config"] == "MB_Shield", "dpd"].std(ddof=1),
        },
        {
            "config": "Fair_Filter",
            "accuracy_mean": results.loc[results["config"] == "Fair_Filter", "accuracy"].mean(),
            "accuracy_std": results.loc[results["config"] == "Fair_Filter", "accuracy"].std(ddof=1),
            "dpd_mean": results.loc[results["config"] == "Fair_Filter", "dpd"].mean(),
            "dpd_std": results.loc[results["config"] == "Fair_Filter", "dpd"].std(ddof=1),
        },
        {
            "config": "Difference (Fair_Filter - MB_Shield)",
            "accuracy_mean": acc_diff.mean(),
            "accuracy_std": acc_diff.std(ddof=1),
            "dpd_mean": dpd_diff.mean(),
            "dpd_std": dpd_diff.std(ddof=1),
            "accuracy_p_value": acc_t.pvalue,
            "dpd_p_value": dpd_t.pvalue,
        }
    ])

    return summary, results


# -----------------------
# EXECUTION (ACSIncome)
# -----------------------
df = load_acs_income_df(survey_year="2018", horizon="1-Year", state='CA')

target_col = "income"
sensitive_col = "SEX"

mb = ["AGEP", "COW", "MAR", "OCCP", "RAC1P", "SCHL", "WKHP", "RELP"]
proxies = ["RELP", "OCCP"]

summary_df, split_results_df = evaluate_tradeoff_repeated_splits(
    df=df,
    target_col=target_col,
    sensitive_col=sensitive_col,
    mb_features=mb,
    proxy_features=proxies,
    n_splits=30,
    test_size=0.30,
    random_state=42,
)

print("\n=== Summary (mean ± std; last row contains p-values) ===")
print(summary_df.round(4).to_string(index=False))

summary_df.to_csv("acs_income_tradeoff_summary.csv", index=False)
split_results_df.to_csv("acs_income_tradeoff_splits.csv", index=False)
print("\nSaved:")
print(" - acs_income_tradeoff_summary.csv")
print(" - acs_income_tradeoff_splits.csv")

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 1 are removed. Consider decreasing the number of bins.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 2 are removed. Consider decreasing the number of bins.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 4 are removed. Consider decreasing the number of bins.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 5 are removed. Consider decreasing the number of bins.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306


=== Summary (mean ± std; last row contains p-values) ===
                              config  accuracy_mean  accuracy_std  dpd_mean  dpd_std  accuracy_p_value  dpd_p_value
                           MB_Shield         0.7920        0.0013    0.0571   0.0031               NaN          NaN
                         Fair_Filter         0.7755        0.0015    0.0593   0.0029               NaN          NaN
Difference (Fair_Filter - MB_Shield)        -0.0165        0.0010    0.0022   0.0026               0.0       0.0001

Saved:
 - acs_income_tradeoff_summary.csv
 - acs_income_tradeoff_splits.csv
